# 🛠️ CIDA SRE Swarm: Interactive Diagnostic & Debug Dashboard

This Jupyter notebook provides a step-by-step diagnostic and execution interface for the **CIDA SRE 72-Hour Autonomous Management Demo**.

---

## 📦 Step 1: Environment Setup & Console Reconfiguration

In [ ]:
import sys
import json
import matplotlib.pyplot as plt

# Ensure console output supports UTF-8 emojis on Windows systems
if hasattr(sys.stdout, 'reconfigure'):
    sys.stdout.reconfigure(encoding='utf-8')

print("✅ Setup complete. System path & graphics engine ready.")

## 🟢 Phase 1: Test Fintech Infrastructure Simulator
We initialize the simulator, print a snapshot of the microservices state, and verify tool routing.

In [ ]:
from cida_sre_demo.simulator import FintechSimulator

sim = FintechSimulator()
print(f"Initial Active Alerts: {sim.get_active_alerts()}")
print(f"Postgres Connections: {sim.get_db_connections()}")
print(f"Service Auth Status: {sim.get_pod_status('auth-service')}")

## 🟡 Phase 2: Load SRE Scenarios
Let's test loading a Level 3 scenario (`SQL Injection Attack`), verify the setup state, and test the validator conditions.

In [ ]:
from cida_sre_demo.scenarios import Scenario11_SQL_Injection

scen = Scenario11_SQL_Injection()
print(f"Loading Scenario {scen.id}: {scen.name} ({scen.level})")
scen.setup(sim)
print(f"Active tokens: {sim.active_tokens}")
print(f"Is scenario resolved initially? {scen.validate(sim)}")

## 🟠 Phase 3: Compare Agents Deliberations (Skeptic vs Prosecutor)
Let's simulate how both agents process the SQL Injection threat side by side.

In [ ]:
from cida_sre_demo.agent_swarm import BaseLLMSREAgent, CidaSwarmSREAgent

print("=== Base LLM Deliberation ===")
agent_base = BaseLLMSREAgent(sim)
for step in agent_base.solve_scenario(11):
    print(step)

print("\n=== CIDA Swarm Deliberation ===")
agent_cida = CidaSwarmSREAgent(sim)
for step in agent_cida.solve_scenario(11):
    print(step)

## 🔴 Phase 4: Run Complete 72-Hour SRE Benchmark
We evaluate both agents on all 20 escalating scenarios sequentially and save the results.

In [ ]:
from cida_sre_demo.run_demo import run_evaluation

run_evaluation()

## 📊 Step 6: Render Performance Diagrams
We load `evaluation_results.json` and generate charts showing Scenarios Solved, Cost, and Reasoning Tokens footprint.

In [ ]:
with open('cida_sre_demo/evaluation_results.json', 'r') as f:
    res = json.load(f)

agents = ['Base LLM (No CIDA)', 'CIDA SRE Swarm']
solved = [res['base_llm']['solved'], res['cida']['solved']]
errors = [res['base_llm']['errors'], res['cida']['errors']]
costs = [res['base_llm']['cost_usd'], res['cida']['cost_usd']]

fig, axs = plt.subplots(1, 2, figsize=(14, 5))

# Solved Scenarios vs Errors
axs[0].bar(agents, solved, color=['#ff6b6b', '#2ecc71'], alpha=0.9)
axs[0].set_title('Scenarios Solved (out of 20)')
axs[0].set_ylabel('Solved Count')
for idx, val in enumerate(solved):
    axs[0].text(idx, val + 0.5, str(val), ha='center', fontweight='bold')

# Cost comparison
axs[1].bar(agents, costs, color=['#f39c12', '#3498db'], alpha=0.9)
axs[1].set_title('Inference cost (USD)')
axs[1].set_ylabel('Cost in USD')
for idx, val in enumerate(costs):
    axs[1].text(idx, val + (val*0.05), f"${val:.4f}", ha='center', fontweight='bold')

plt.suptitle('CIDA-SRE vs Base LLM: 72-Hour Autonomous SRE Evaluation', fontsize=14, fontweight='bold')
plt.tight_layout()
plt.show()